# 算子调试与性能调优

## 概述

在前五章中，我们已经完成了从核函数基础、Vector算子、Matmul高阶API到融合算子的开发学习。但算子开发的完整流程还包括两个关键阶段：**写对**（功能调试）与**写快**（性能调优）。

- **功能调试**解决"结果不对"的问题：多核切分越界、偏移计算错误、数据搬运错位等，需要在不方便直接断点的Device侧代码中定位问题。
- **性能调优**解决"运行慢"的问题：算子逻辑正确但耗时远超预期，需要量化定位瓶颈在搬运、计算还是流水组织，再做针对性优化。

调试调优是算子开发从"能跑"到"能用"的最后一公里。本章以Add算子为贯穿案例：6.2从一个"结果错误"的Add算子出发，学习`asc.printf`与`asc.dump_tensor`两大调试接口；6.3从一个"运行正确但慢"的Add算子出发，学习msprof op上板性能采集，走完"采集→分析→优化→复测"的完整闭环。

## 两大场景与工具选择

| 问题特征 | 典型现象 | 首选工具 |
| --- | --- | --- |
| 结果错误 | `torch.allclose`断言失败、输出含NaN、部分数据错乱 | `asc.printf` / `asc.dump_tensor`（6.2） |
| 运行正确但耗时高 | 算子逻辑无误，但耗时远超理论值 | msprof op 上板采集（6.3） |

## 学习前置要求

在开始本章学习前，请确认你已具备以下能力：

| 类别 | 要求 |
| --- | --- |
| **已具备知识** | 掌握pyasc编程范式与核函数开发（第2章Add算子）；理解手动流水同步`set_flag`/`wait_flag`与双缓冲原理（2.5节）；了解Matmul基础API（第4章） |
| **环境要求** | CANN 8.5.0.alpha001及以上；pyasc v1.1.0及以上；msprof随CANN工具链安装 |

## 学习目标

完成本章后，你将能够：

1. **掌握调试接口**：熟练使用`asc.printf`打印标量与执行路径信息，使用`asc.dump_tensor`打印GM/UB中的Tensor数据，并了解两接口的使用约束与性能影响。
2. **掌握调试方法论**：按照"现象→假设→取证→定位→修复→复测"六步流程系统定位算子功能问题。
3. **掌握性能数据采集与分析**：使用msprof op采集上板性能数据，用pandas读取分析CSV指标，判定性能瓶颈。
4. **掌握完整调优闭环**：完成"劣化算子→瓶颈分析→针对性优化→复测对比"的端到端调优实践。

## 章节内容导航

| 小节 | 主题 | 核心内容 |
| --- | --- | --- |
| [6.1 章节概述](./06.01_chapter_intro.ipynb) | 本章导览 | 学习前置要求、目标、内容导航 |
| [6.2 功能调试](./06.02_functional_debugging.ipynb) | printf/dump_tensor | 调试接口原理、参数详解、Add算子调试实战、调试方法论 |
| [6.3 性能调优](./06.03_performance_profiling.ipynb) | msprof op | 上板采集与CSV分析、劣化→优化实战 |
| [6.4 章节实践](./06.04_chapter_practice.ipynb) | 综合实践 | Mul算子"先修复后提速"双任务闭环 |

## 运行环境与硬件说明

### 支持硬件

| 硬件型号 | 架构 | 支持状态 |
| --- | --- | --- |
| Atlas A2（910B） | dav-matrix-core | ✅ 完全支持 |
| Atlas A3（910C） | dav-matrix-core | ✅ 完全支持 |

### 工具说明

- **msprof op**：随CANN工具链安装，位于CANN安装路径的tools目录下，需先执行`source ${ASCEND_TOOLKIT_HOME}/set_env.sh`使环境变量生效。
